# Week 5 – Spark DataFrames: Cleaning, Transformation and Aggregation

The dataset here is built on top of the Sample Superstore CSV used in weeks 2 and 3.
Superstore gives us real regions, cities, categories and sales, but the questions this
week also mention columns it does not have (age, subscription, status, email, username,
store_id, raw_timestamp), so `data/prepare_dataset.py` keeps every Superstore row,
derives the missing columns, and injects the mess the cleaning steps are meant to deal
with: duplicate rows, null prices and statuses, blank usernames, missing emails and a
few timestamps in a broken format.

Everything runs on a local Spark session (PySpark 4.2).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder.appName("week5_dataframes").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("../data/transactions.csv", header=True, inferSchema=True)
print(df.count(), "rows")
df.printSchema()

10494 rows
root
 |-- transaction_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)



In [2]:
df.select("transaction_id", "user_id", "transaction_date", "region", "city",
          "product_category", "price", "quantity", "sale_amount", "status").show(5)

+--------------+--------+----------------+-------+-------------+----------------+------+--------+-----------+---------+
|transaction_id| user_id|transaction_date| region|         city|product_category| price|quantity|sale_amount|   status|
+--------------+--------+----------------+-------+-------------+----------------+------+--------+-----------+---------+
|        T10881|HP-14815|      2017-11-30|   East|New York City|       Furniture|248.58|       5|     1242.9|   Failed|
|        T19319|AR-10825|      2017-07-20|   East|New York City| Office Supplies|  6.68|       2|      13.36|   Failed|
|        T15620|SC-20695|      2016-09-25|Central|    La Crosse|      Technology| 99.99|       5|     499.95|Completed|
|        T19062|CS-12250|      2015-11-20|   West|    San Diego| Office Supplies| 12.53|       4|      50.12|Completed|
|        T10954|GH-14665|      2017-12-28|Central|   Round Rock| Office Supplies| 13.58|       2|     27.168|Completed|
+--------------+--------+---------------

## Q1 – Why Spark over traditional MapReduce?

The core problem with MapReduce is that it treats disk as the hand-off point between
every stage. Each map phase writes its intermediate output to disk, the reduce phase
reads it back, and if your job logically needs several stages you end up chaining
multiple full MapReduce jobs, each one paying the write-to-HDFS / read-from-HDFS tax
plus job scheduling overhead. The other pain points:

- **Rigid model.** Everything has to be forced into a map step and a reduce step, even
  when the problem is naturally a join, a filter or a multi-stage flow. Code gets long
  and awkward.
- **No good story for iteration.** Algorithms that loop over the same data (ML training,
  PageRank) re-read the input from disk on every single pass.
- **Batch only.** There is no interactive querying – latency is minutes, not seconds.

Spark fixes this with a DAG execution engine that plans the whole job as one graph,
keeps intermediate data in memory across stages, and exposes a high-level DataFrame/SQL
API. One platform also covers SQL, streaming and MLlib instead of needing separate
tools bolted onto Hadoop.

## Q2 – In-memory computing and iterative ML

Iterative algorithms (gradient descent, k-means, ALS...) make many passes over the same
training data – often dozens or hundreds. On a disk-based system like MapReduce, every
iteration is a fresh job that reads the full dataset from HDFS, computes, and writes
results back to HDFS. The actual math is usually cheap; the disk I/O dominates.

Spark loads the dataset once and pins it in executor memory with `.cache()` /
`.persist()`. Iteration 1 pays the disk read, iterations 2 through N read from RAM,
which is orders of magnitude faster. That is where the often-quoted 10–100x speedups
on ML workloads come from. If an executor dies, Spark does not need a full restart
either – each DataFrame carries its lineage, so only the lost partitions get recomputed.

## Q3 – Removing duplicates on user_id + transaction_date

`dropDuplicates` with a column subset keeps the first row it encounters for each
distinct (user_id, transaction_date) pair and drops the rest. Worth noting how
aggressive this key is on this particular dataset: a Superstore order has several line
items, all sharing the same customer and date, so this collapses far more than just the
injected duplicates. That is the real lesson here – the dedup key has to match what you
consider "the same record".

In [3]:
before = df.count()
df_dedup = df.dropDuplicates(["user_id", "transaction_date"])
after = df_dedup.count()
print(f"{before} rows -> {after} rows ({before - after} removed)")

10494 rows -> 4992 rows (5502 removed)


## Q4 – Average sale amount per category in the West

Filter first, then group – Spark pushes the filter before the shuffle, so only West
rows get moved around.

In [4]:
df.filter(F.col("region") == "West") \
  .groupBy("product_category") \
  .agg(F.round(F.avg("sale_amount"), 2).alias("avg_sale_amount")) \
  .orderBy(F.desc("avg_sale_amount")) \
  .show()

+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
|      Technology|         414.36|
|       Furniture|         356.19|
| Office Supplies|         116.57|
+----------------+---------------+



## Q5 – `.na.drop()` vs `.na.fill()`

They answer two different questions about rows with nulls:

- `.na.drop()` **removes** rows. By default any null in any column kills the row;
  `how="all"` and `subset=[...]` narrow it down. Use it when a row without that value
  is useless.
- `.na.fill()` **keeps** rows and replaces the nulls with a value you choose. Use it
  when the row still carries information and you just need a sensible default.

Both return new DataFrames – nothing is modified in place. Filling the `status` nulls
with `'Unknown'`:

In [5]:
print("null statuses before:", df.filter(F.col("status").isNull()).count())

df_status = df.na.fill({"status": "Unknown"})
df_status.groupBy("status").count().orderBy(F.desc("count")).show()

null statuses before: 627


+---------+-----+
|   status|count|
+---------+-----+
|   Failed| 3404|
|  Pending| 3239|
|Completed| 3224|
|  Unknown|  627|
+---------+-----+



## Q6 – Cities with more than 100 records

A classic HAVING-style query: aggregate first, then filter on the aggregated value.
The condition goes on the result of `count()`, not on the raw rows.

In [6]:
df.groupBy("city").count() \
  .filter(F.col("count") > 100) \
  .orderBy(F.desc("count")) \
  .show()

+-------------+-----+
|         city|count|
+-------------+-----+
|New York City|  967|
|  Los Angeles|  781|
| Philadelphia|  568|
|San Francisco|  542|
|      Seattle|  452|
|      Houston|  395|
|      Chicago|  333|
|     Columbus|  233|
|    San Diego|  176|
|  Springfield|  170|
|       Dallas|  168|
| Jacksonville|  129|
|      Detroit|  122|
+-------------+-----+



## Q7 – Immutability and data cleaning

A Spark DataFrame can never be edited in place. "Dropping" or "renaming" a column does
not touch the original – it returns a *new* DataFrame with the change applied, and the
old one is still there, untouched, until it goes out of scope. In practice that shapes
how cleaning code looks:

- every step is an assignment: `df_clean = df.drop("col")`, and forgetting the
  assignment is the classic beginner bug (the line runs fine and does nothing),
- steps chain naturally into pipelines, which is exactly what Q15 does,
- you can keep the raw DataFrame around and compare before/after for free,
- since transformations are lazy, the chain costs nothing until an action forces it –
  Spark records lineage and optimises the whole chain at once.

Quick proof that the original survives a rename:

In [7]:
renamed = df.withColumnRenamed("sale_amount", "revenue")
print("original columns still contain sale_amount:", "sale_amount" in df.columns)
print("new DataFrame has revenue instead:        ", "revenue" in renamed.columns)

original columns still contain sale_amount: True
new DataFrame has revenue instead:         True


## Q8 – Premium subscribers aged 18 to 30

`between` is inclusive on both ends, which is exactly what the question asks for.

In [8]:
premium_young = df.filter(F.col("age").between(18, 30) & (F.col("subscription") == "Premium"))
print(premium_young.count(), "matching rows")
premium_young.select("user_id", "age", "subscription", "product_category", "sale_amount").show(5)

860 matching rows


+--------+---+------------+----------------+-----------+
| user_id|age|subscription|product_category|sale_amount|
+--------+---+------------+----------------+-----------+
|PK-19075| 28|     Premium|       Furniture|     242.94|
|DL-12865| 28|     Premium|       Furniture|     15.384|
|CS-12400| 24|     Premium| Office Supplies|       2.48|
|TB-21055| 22|     Premium| Office Supplies|     208.56|
|MC-17605| 28|     Premium|       Furniture|     34.504|
+--------+---+------------+----------------+-----------+
only showing top 5 rows


## Q9 – Why handle nulls before aggregating?

Because Spark's aggregations skip nulls *silently*, and that quietly changes what your
numbers mean. `avg()` divides by the count of non-null values, not the row count, so a
column that is 5% null gives you an average over 95% of the data without any warning.
`sum()` over a group that is entirely null returns null, not 0. And in a `groupBy`,
null keys form their own group.

None of that is wrong, but it should be a decision, not an accident. Handling nulls
first (drop them, or fill with 0 / a mean / a flag value) makes the denominator
explicit. The demo below shows the same column giving two different averages depending
on that choice – filling with 0 pulls the average down because the denominator grows:

In [9]:
df.agg(
    F.count("*").alias("total_rows"),
    F.count("price").alias("non_null_prices"),
    F.round(F.avg("price"), 2).alias("avg_ignoring_nulls")
).show()

df.na.fill({"price": 0}).agg(
    F.round(F.avg("price"), 2).alias("avg_after_filling_0")
).show()

+----------+---------------+------------------+
|total_rows|non_null_prices|avg_ignoring_nulls|
+----------+---------------+------------------+
|     10494|           9991|             60.85|
+----------+---------------+------------------+



+-------------------+
|avg_after_filling_0|
+-------------------+
|              57.94|
+-------------------+



## Q10 – Cast raw_timestamp to TimestampType and rename to event_time

A note from actually running this: my first attempt used a plain
`.cast(TimestampType())` and the job crashed with `CAST_INVALID_INPUT` on
`'17/9/2015 19.27'`. Spark 4 enables ANSI mode by default, so a malformed value now
fails the whole job instead of quietly becoming null the way it did in Spark 3.
`try_cast` restores the tolerant behaviour explicitly – bad values become null and
you can count them afterwards, which is what I want here.

In [10]:
df_ts = df.withColumn("raw_timestamp", F.col("raw_timestamp").try_cast(TimestampType())) \
          .withColumnRenamed("raw_timestamp", "event_time")

df_ts.select("event_time").printSchema()
df_ts.select("event_time").show(3)
print("rows where the cast failed and produced null:",
      df_ts.filter(F.col("event_time").isNull()).count())

root
 |-- event_time: timestamp (nullable = true)



+-------------------+
|         event_time|
+-------------------+
|2017-11-30 19:28:59|
|2017-07-20 15:51:35|
|2016-09-25 08:43:43|
+-------------------+
only showing top 3 rows
rows where the cast failed and produced null: 114


The handful of timestamps that arrived in `d/m/yyyy h.m` format do not match what
the cast expects, so they end up as null – visible, countable, and exactly the trap
Q14 talks about.

## Q11 – The shuffle, and why groupBy is a wide transformation

Rows for the same key are scattered across partitions – there is no reason all the
`S07` rows would sit together. To group them, Spark has to physically move data: each
task hashes the key, writes its rows into per-partition shuffle files on local disk,
and the next stage pulls its share of every one of those files over the network. That
whole exchange is the shuffle.

It is called a **wide** transformation because one output partition depends on *many*
input partitions (contrast `filter` or `withColumn`, where each output partition comes
from exactly one input partition – narrow, no data movement). The shuffle is the
expensive part of a Spark job: disk writes, network transfer, serialization, and it
forces a stage boundary in the DAG. You can see it in the physical plan below as the
`Exchange hashpartitioning` step:

In [11]:
df.groupBy("store_id").agg(F.sum("sale_amount")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[store_id#30], functions=[sum(sale_amount#26)])
   +- Exchange hashpartitioning(store_id#30, 200), ENSURE_REQUIREMENTS, [plan_id=609]
      +- HashAggregate(keys=[store_id#30], functions=[partial_sum(sale_amount#26)])
         +- FileScan csv [sale_amount#26,store_id#30] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/USER/Desktop/CelebalAssignments/week5/data/transactions..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_amount:double,store_id:string>




## Q12 – Drop rows with null email or blank username

One thing the raw data taught me here: Spark's CSV reader already turns truly empty
fields into nulls on read, so the missing emails show up as null. The usernames are
worse – they come in as whitespace, which is not null and not equal to `""` either, so
`trim` is needed to catch them. The filter keeps rows where email is present AND the
trimmed username is non-empty, which is the complement of "email is null OR username is
empty".

In [12]:
before = df.count()
df_contacts = df.filter(F.col("email").isNotNull() & (F.trim(F.col("username")) != ""))
after = df_contacts.count()
print(f"{before} rows -> {after} rows ({before - after} removed)")

10494 rows -> 9822 rows (672 removed)


## Q13 – Several statistics in one `.agg()` call

`.agg()` takes any number of aggregate expressions, so one pass over the data returns
all three. The same works after a `groupBy` for per-group stats.

In [13]:
df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.round(F.avg("price"), 2).alias("mean_price")
).show()

df.groupBy("product_category").agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.round(F.avg("price"), 2).alias("mean_price")
).show()

+---------+---------+----------+
|min_price|max_price|mean_price|
+---------+---------+----------+
|     0.34|  3773.08|     60.85|
+---------+---------+----------+



+----------------+---------+---------+----------+
|product_category|min_price|max_price|mean_price|
+----------------+---------+---------+----------+
| Office Supplies|     0.34|  1889.99|     32.27|
|       Furniture|     1.16|   880.98|     90.27|
|      Technology|     0.79|  3773.08|    120.15|
+----------------+---------+---------+----------+



## Q14 – The risk of `inferSchema=true` on messy dates

Schema inference guesses each column's type from the data itself, and with inconsistent
date formats that guess goes wrong in one of two ways:

1. **The column falls back to string.** If the scan sees mixed formats, inference gives
   up and types the column as string. Nothing fails loudly – but every date comparison,
   sort or window downstream is now string logic, and `'11/8/2016' < '2/1/2015'` is
   true alphabetically.
2. **Silent nulls.** The nastier case: the values inference looked at happen to be
   clean, so the column gets typed as date/timestamp – and then every row in a
   different format silently becomes null in permissive mode. Data loss with no error.

This notebook hit case 1 for real: `raw_timestamp` has about 1% of rows in
`d/m/yyyy h.m` format, so inference typed it string, and the explicit cast in Q10
nulled those rows. In production the honest approach is an explicit schema, reading
messy date columns as string, and converting deliberately with `to_timestamp` and known
format patterns so failures are visible and countable.

## Q15 – Full pipeline: dedup, fill null prices, revenue per store

For the dedup step I treat rows that are identical in everything except
`transaction_id` as duplicates – that catches both the exact double-submits and the
retried payments that came back with a fresh id. Then null prices become 0 (a missing
price should not inflate averages, and for a revenue *sum* zero is the conservative
choice), revenue is price x quantity, and everything rolls up per store.

In [14]:
line_cols = [c for c in df.columns if c != "transaction_id"]

revenue_per_store = (
    df.dropDuplicates(line_cols)
      .na.fill({"price": 0})
      .withColumn("revenue", F.round(F.col("price") * F.col("quantity"), 2))
      .groupBy("store_id")
      .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
           F.count("*").alias("transactions"))
      .orderBy(F.desc("total_revenue"))
)
revenue_per_store.show(12)

revenue_per_store.toPandas().to_csv("../output/store_revenue.csv", index=False)

+--------+-------------+------------+
|store_id|total_revenue|transactions|
+--------+-------------+------------+
|     S02|    199804.87|         880|
|     S10|    198465.49|         836|
|     S07|    197539.52|         823|
|     S12|    190626.76|         826|
|     S09|    190167.42|         833|
|     S08|    187460.44|         873|
|     S04|    183631.81|         789|
|     S11|    171299.34|         822|
|     S06|    170782.22|         803|
|     S01|    170644.36|         873|
|     S03|    166028.37|         826|
|     S05|    159551.73|         810|
+--------+-------------+------------+



## Insights

- **The dedup key is a modelling decision, not a syntax detail.** Deduplicating on
  (user_id, transaction_date) collapsed 10,494 rows to 4,992 – far more than the
  injected duplicates – because a Superstore order has several line items that all
  share a customer and a date. The pipeline in Q15 instead treats rows identical in
  everything but transaction_id as duplicates, which only removes the double-submits
  and retried payments it should.
- **Nulls quietly change averages.** The average price is 60.85 when the 503 missing
  values are skipped (Spark's default) but 57.94 after filling them with 0. Neither
  number is wrong, but they answer different questions, which is why null handling has
  to happen consciously before aggregating.
- **Spark 4's ANSI mode changes old habits.** A plain `.cast(TimestampType())`
  crashed the job on the first malformed timestamp; `try_cast` converted the 114 bad
  rows to nulls that can be counted and dealt with.
- **West region mirrors the overall Superstore pattern** – Technology has the highest
  average sale (414.36), Furniture sits in the middle (356.19), and Office Supplies is
  cheap and frequent (116.57).
- 13 cities have more than 100 transactions; New York City (967) and Los Angeles (781)
  dominate. 627 transactions (about 6%) had no status and now sit in an explicit
  'Unknown' bucket rather than a null one.
- **Store revenue comes out remarkably even** (S02 tops at ~199.8k, S05 trails at
  ~159.6k). That is expected: store_id was assigned randomly when the dataset was
  built, so the value of Q15 is in the pipeline shape, not the ranking itself.

In [15]:
spark.stop()